# Notebook 6: Random Forests

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 3 hours | **Week 9 — Tree-Based Methods & Gradient Boosting**

---

## Why This Matters

Random Forests are one of the most robust, well-understood, and widely-deployed algorithms in machine learning. They require minimal tuning, handle mixed feature types naturally, are resistant to overfitting, produce a free uncertainty estimate (OOB error), and generate reliable feature importance scores. Understanding *why* they work — especially the role of decorrelation through random feature subsets — gives you the intuition to debug failures, tune hyperparameters meaningfully, and understand XGBoost and LightGBM (which Random Forests prefigured).

---

## The Independent Judges Analogy

A panel of 500 judges independently evaluates a house. Each judge:
- Studies a **different sample** of past house sales (bootstrap)
- Is only allowed to look at **3 randomly chosen features** out of 8 available (e.g., one judge only sees sqft, age, and garage; another sees bedrooms, school score, and distance)
- Makes their estimate based solely on those 3 features

The panel then averages all 500 estimates.

Why is this better than a single expert who sees all features and all data? Because the judges' **errors are uncorrelated**:
- Different training samples → different idiosyncratic errors (bootstrap)
- Different feature sets → even when one feature dominates (e.g., sqft is very important), not every judge can use it → each judge finds different splits → uncorrelated predictions

When uncorrelated errors are averaged, they cancel. The panel's wisdom exceeds any individual judge.

## Random Forest = Bagging + Random Feature Subsets

Plain bagging already helps: each tree sees a bootstrap sample → slightly different trees → averaging reduces variance. But there is a problem: if feature 1 is very predictive (say, sqft strongly determines price), **every tree will split on feature 1 at the root**. Despite different bootstrap samples, all trees make very similar predictions → high correlation → limited benefit from averaging.

Random forests break this correlation with a second source of randomness:

> **At each split in each tree**, only a random subset of $m$ features is considered (not all $p$).

If sqft is dominant but a tree is not allowed to see it, it must find the next-best split using whichever features it does have. Different trees find different structures. Their predictions are less correlated.

**Default values:**
- Regression: $m = \lfloor p/3 \rfloor$ (or `max_features='sqrt'` in sklearn)
- Classification: $m = \lfloor \sqrt{p} \rfloor$

## Mathematical Intuition

If $\rho$ = pairwise correlation between tree predictions and $\sigma^2$ = single-tree prediction variance:

$$\boxed{\text{Ensemble Variance} = \rho \sigma^2 + \frac{(1-\rho)\sigma^2}{B}}$$

- As $B \to \infty$: Variance $\to \rho \sigma^2$ (the **variance floor**)
- **Bagging only** (no feature subsets): $\rho$ is moderate (important features dominate every tree)
- **Random Forest** (feature subsets): $\rho$ is lower → lower variance floor → better ensemble

Random feature subsets reduce $\rho$ at the cost of increasing $\sigma^2$ slightly (each individual tree is weaker). But for $B$ large enough, the lower $\rho$ wins.

## Out-of-Bag (OOB) Error

Each bootstrap sample leaves out approximately $1 - (1 - 1/n)^n \approx 1 - e^{-1} \approx 36.8\%$ of the training data on average. These are the **out-of-bag (OOB) samples** for that tree.

For each training point $x_i$:
- Identify the trees that did **not** use $x_i$ in their bootstrap sample
- Average those trees' predictions for $x_i$
- Compare to $y_i$

This gives a free validation error estimate — no need to hold out a separate validation set. OOB error is an approximately unbiased estimate of test error, comparable to 5-fold or 10-fold cross-validation, but computed at zero extra training cost.

In sklearn: `RandomForestRegressor(oob_score=True)` → access via `model.oob_score_` (R²) or compute RMSE manually from `model.oob_prediction_`.

## Feature Importance in Random Forests

Random forests give two types of feature importance:

1. **Impurity-based (Mean Decrease Impurity / MDI):** Average reduction in impurity (MSE for regression) across all splits on that feature, weighted by the number of samples. Fast, built-in, but can favour high-cardinality features.

2. **Permutation importance:** Shuffle one feature at a time; measure how much OOB error increases. More reliable, but slower.

Both are accessible in sklearn: `model.feature_importances_` (MDI) and `sklearn.inspection.permutation_importance`.

In [ ]:
# ============================================================
# Cell 1: Imports and Global Setup
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print('Libraries loaded. NumPy seed set to 42.')

In [ ]:
# ============================================================
# Cell 2: Synthetic House Price Dataset (1000 samples, 8 features)
# ============================================================
np.random.seed(42)
n_samples = 1000

# Feature generation
sqft          = np.random.normal(1900, 550, n_samples).clip(400, 5000)
bedrooms      = np.random.randint(1, 7, n_samples).astype(float)
bathrooms     = np.random.randint(1, 5, n_samples).astype(float)
age           = np.random.uniform(0, 60, n_samples)
distance_cbd  = np.random.uniform(1, 35, n_samples)   # km
garage        = np.random.randint(0, 4, n_samples).astype(float)
school_score  = np.random.uniform(3, 10, n_samples)
lot_size      = np.random.normal(600, 200, n_samples).clip(100, 2000)  # m²

# Price formula with nonlinearities and interactions
price = (
      85 * sqft
    + 14000 * bedrooms
    + 18000 * bathrooms
    - 1500 * age
    - 5500 * distance_cbd
    + 20000 * garage
    + 13000 * school_score
    + 80 * lot_size
    + 0.018 * sqft**2           # nonlinear sqft
    - 250 * age * distance_cbd  # interaction
    + 5000 * (bedrooms * bathrooms)  # interaction
    + np.random.normal(0, 28000, n_samples)
) / 1000

price = price.clip(80, 1200)

# DataFrame
FEATURES = ['sqft', 'bedrooms', 'bathrooms', 'age',
            'distance_cbd', 'garage', 'school_score', 'lot_size']

df = pd.DataFrame({
    'sqft': sqft, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
    'age': age, 'distance_cbd': distance_cbd, 'garage': garage,
    'school_score': school_score, 'lot_size': lot_size,
    'price_k': price
})

median_price = df['price_k'].median()
df['above_median'] = (df['price_k'] > median_price).astype(int)

print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Features: {FEATURES}')
print(f'Price range: ${df.price_k.min():.0f}k – ${df.price_k.max():.0f}k')
print(f'Median price: ${median_price:.0f}k')
df.describe().round(1)

In [ ]:
# ============================================================
# Cell 3: Train/Test Split
# ============================================================
X = df[FEATURES].values
y = df['price_k'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Features: p = {X_train.shape[1]}')
print(f'Default max_features (sqrt): {int(np.sqrt(X_train.shape[1]))}')

In [ ]:
# ============================================================
# Cell 4: Random Forest from Scratch
# ============================================================
class RandomForestScratch:
    """
    Random Forest regressor implemented from first principles.
    Demonstrates: bootstrap sampling + random feature subsets per split.
    """
    def __init__(self, n_estimators=100, max_depth=5,
                 max_features='sqrt', random_state=42):
        self.n_estimators  = n_estimators
        self.max_depth     = max_depth
        self.max_features  = max_features
        self.random_state  = random_state
        self.trees         = []
        self.feature_indices = []  # which features each tree used

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n, p = X.shape

        # Determine number of features to sample per split
        if self.max_features == 'sqrt':
            n_features = max(1, int(np.sqrt(p)))
        elif self.max_features == 'log2':
            n_features = max(1, int(np.log2(p)))
        elif isinstance(self.max_features, float):
            n_features = max(1, int(self.max_features * p))
        else:
            n_features = int(self.max_features)

        for i in range(self.n_estimators):
            # 1. Bootstrap sample
            boot_idx = np.random.choice(n, n, replace=True)
            X_boot   = X[boot_idx]
            y_boot   = y[boot_idx]

            # 2. Random feature subset for THIS tree
            feat_idx = np.random.choice(p, n_features, replace=False)
            self.feature_indices.append(feat_idx)

            # 3. Train a deep decision tree on the selected features
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                random_state=i  # each tree sees different random splits too
            )
            tree.fit(X_boot[:, feat_idx], y_boot)
            self.trees.append(tree)

        return self

    def predict(self, X):
        # Average predictions across all trees
        all_preds = np.array([
            tree.predict(X[:, feat_idx])
            for tree, feat_idx in zip(self.trees, self.feature_indices)
        ])
        return all_preds.mean(axis=0)

    def predict_all_trees(self, X):
        """Return predictions of all individual trees (shape: n_estimators x n_samples)."""
        return np.array([
            tree.predict(X[:, feat_idx])
            for tree, feat_idx in zip(self.trees, self.feature_indices)
        ])


# Instantiate and train
rf_scratch = RandomForestScratch(
    n_estimators=100, max_depth=None, max_features='sqrt', random_state=RANDOM_STATE
)
rf_scratch.fit(X_train, y_train)

rmse_scratch = rmse(y_test, rf_scratch.predict(X_test))
print(f'RandomForestScratch RMSE: {rmse_scratch:.2f}$k')
print(f'Number of trees trained: {len(rf_scratch.trees)}')
print(f'Features per tree: {len(rf_scratch.feature_indices[0])} (out of {X_train.shape[1]})')

In [ ]:
# ============================================================
# Cell 5: Side-by-Side RMSE Comparison
#   Single Tree | Scratch RF | sklearn RF
# ============================================================
# Single decision tree (full depth)
single_tree = DecisionTreeRegressor(max_depth=None, random_state=RANDOM_STATE)
single_tree.fit(X_train, y_train)
rmse_single = rmse(y_test, single_tree.predict(X_test))

# sklearn Random Forest
rf_sklearn = RandomForestRegressor(
    n_estimators=100, max_depth=None, max_features='sqrt',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_sklearn.fit(X_train, y_train)
rmse_sklearn = rmse(y_test, rf_sklearn.predict(X_test))

# Plain bagging (no feature subsets) for comparison
bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=None),
    n_estimators=100, bootstrap=True,
    random_state=RANDOM_STATE, n_jobs=-1
)
bag.fit(X_train, y_train)
rmse_bag = rmse(y_test, bag.predict(X_test))

print('RMSE Comparison (Test Set):')
print(f'  Single Decision Tree:        {rmse_single:.2f}$k')
print(f'  Bagging only (100 trees):    {rmse_bag:.2f}$k')
print(f'  Random Forest Scratch (100): {rmse_scratch:.2f}$k')
print(f'  sklearn RandomForest (100):  {rmse_sklearn:.2f}$k')
print()
print(f'Random Forest improvement over Bagging: {100*(rmse_bag - rmse_scratch)/rmse_bag:.1f}%')
print('(Feature subsets decorrelate trees, reducing variance below plain bagging.)')

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
methods = ['Single Tree', 'Bagging\n(100 trees)', 'RF Scratch\n(100 trees)', 'sklearn RF\n(100 trees)']
rmses   = [rmse_single, rmse_bag, rmse_scratch, rmse_sklearn]
colors  = ['#e74c3c', '#e67e22', '#3498db', '#27ae60']
bars = ax.bar(methods, rmses, color=colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
            f'{val:.1f}$k', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Test RMSE ($k)')
ax.set_title('Single Tree → Bagging → Random Forest\nEach step reduces variance further', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 6: n_estimators Effect — 1 to 300 Trees
#   Plot validation RMSE curve showing stabilisation
# ============================================================
estimator_range = list(range(1, 11)) + list(range(15, 51, 5)) + list(range(60, 301, 20))
cv_rmses = []

for n_est in estimator_range:
    rf = RandomForestRegressor(
        n_estimators=n_est, max_depth=None, max_features='sqrt',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    scores = cross_val_score(
        rf, X_train, y_train,
        cv=3, scoring='neg_root_mean_squared_error'
    )
    cv_rmses.append(-scores.mean())

# Find approximate stabilisation point
improvements = -np.diff(cv_rmses)
stable_i = next((i for i, imp in enumerate(improvements[5:], 5)
                  if imp < 0.1), len(estimator_range) - 1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(estimator_range, cv_rmses, marker='o', markersize=4,
        color='#27ae60', linewidth=2.2, label='CV RMSE (3-fold)')
ax.axhline(cv_rmses[-1], color='#e74c3c', linestyle='--', alpha=0.7,
           label=f'Plateau ≈ {cv_rmses[-1]:.2f}$k')
ax.axvline(estimator_range[stable_i], color='#8e44ad', linestyle=':',
           linewidth=2, label=f'Stabilises ≈ {estimator_range[stable_i]} trees')

ax.set_xlabel('n_estimators (number of trees)', fontsize=12)
ax.set_ylabel('CV RMSE ($k)', fontsize=12)
ax.set_title('Random Forest: Validation RMSE vs Number of Trees\n'
             'More trees always helps (or at worst stays flat) — never hurts', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

print(f'Key insight: RMSE drops sharply up to ~{estimator_range[stable_i]} trees, then plateaus.')
print('Unlike boosting, more trees NEVER causes overfitting in a random forest.')
print('The practical question is only: "Are the extra trees worth the compute cost?"')

In [ ]:
# ============================================================
# Cell 7: Out-of-Bag (OOB) Error
#   Compare OOB error vs cross-validation error
# ============================================================
rf_oob = RandomForestRegressor(
    n_estimators=200, max_depth=None, max_features='sqrt',
    oob_score=True, random_state=RANDOM_STATE, n_jobs=-1
)
rf_oob.fit(X_train, y_train)

# OOB RMSE: computed from oob_prediction_ (available after fit)
oob_rmse = rmse(y_train, rf_oob.oob_prediction_)

# Cross-validation RMSE (5-fold, for comparison)
cv_scores = cross_val_score(
    RandomForestRegressor(
        n_estimators=200, max_depth=None, max_features='sqrt',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    X_train, y_train, cv=5,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_5fold = -cv_scores.mean()
cv_rmse_std   = cv_scores.std()

# Held-out test RMSE
test_rmse_oob = rmse(y_test, rf_oob.predict(X_test))

print('Out-of-Bag Error vs Cross-Validation Error:')
print(f'  OOB RMSE (free!):          {oob_rmse:.2f}$k')
print(f'  5-fold CV RMSE:            {cv_rmse_5fold:.2f}$k  (±{cv_rmse_std:.2f})')
print(f'  True test RMSE:            {test_rmse_oob:.2f}$k')
print()
print(f'OOB vs CV difference:       {abs(oob_rmse - cv_rmse_5fold):.2f}$k')
print()
print('OOB error is a free, approximately unbiased estimate of test error.')
print('You get it at NO extra computation cost — each OOB sample is predicted')
print('by trees that never trained on it, just like a held-out test set.')

# Scatter: OOB predictions vs true training labels
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Out-of-Bag Predictions vs True Labels', fontsize=13, fontweight='bold')

axes[0].scatter(y_train, rf_oob.oob_prediction_, alpha=0.3, s=10, color='#2980b9')
lo, hi = y_train.min(), y_train.max()
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('True Price ($k)')
axes[0].set_ylabel('OOB Prediction ($k)')
axes[0].set_title(f'OOB Predictions (RMSE = {oob_rmse:.1f}$k)')
axes[0].legend()

# Bar chart comparison
labels = ['OOB RMSE\n(free)', '5-fold CV\nRMSE', 'Test RMSE']
values = [oob_rmse, cv_rmse_5fold, test_rmse_oob]
colors_ = ['#27ae60', '#3498db', '#e74c3c']
bars = axes[1].bar(labels, values, color=colors_, edgecolor='white', width=0.45)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.2,
                 f'{val:.1f}$k', ha='center', fontsize=10)
axes[1].set_ylabel('RMSE ($k)')
axes[1].set_title('OOB ≈ CV ≈ Test Error\n(All estimate the same quantity)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 8: OOB Error as n_estimators Grows
#   Demonstrates OOB stabilisation — when can you stop adding trees?
# ============================================================
print('Tracking OOB RMSE as trees are added...')
n_est_range = list(range(10, 301, 10))
oob_rmses_track = []

for n_est in n_est_range:
    rf_t = RandomForestRegressor(
        n_estimators=n_est, max_depth=None, max_features='sqrt',
        oob_score=True, random_state=RANDOM_STATE, n_jobs=-1, warm_start=False
    )
    rf_t.fit(X_train, y_train)
    oob_rmses_track.append(rmse(y_train, rf_t.oob_prediction_))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_est_range, oob_rmses_track, color='#8e44ad', linewidth=2.2,
        marker='o', markersize=4, label='OOB RMSE')
ax.axhline(min(oob_rmses_track), color='black', linestyle='--', alpha=0.5,
           label=f'Min OOB = {min(oob_rmses_track):.2f}$k')
ax.set_xlabel('n_estimators', fontsize=12)
ax.set_ylabel('OOB RMSE ($k)', fontsize=12)
ax.set_title('OOB Error Tracking — When Do More Trees Stop Helping?\n'
             'Stop adding trees when OOB RMSE plateaus', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
print('OOB error can be used as a stopping criterion — no separate validation set needed!')

In [ ]:
# ============================================================
# Cell 9: max_features Effect
#   Compare: sqrt, log2, 0.3, 0.5, 1.0 (all features)
# ============================================================
p = X_train.shape[1]  # 8 features

max_features_configs = {
    f'sqrt ({int(np.sqrt(p))}/{p})':  'sqrt',
    f'log2 ({int(np.log2(p))}/{p})':  'log2',
    f'30% ({int(0.3*p)}/{p})':        0.3,
    f'50% ({int(0.5*p)}/{p})':        0.5,
    f'All ({p}/{p}) — plain bagging': 1.0,
}

mf_results = {}
for label, mf in max_features_configs.items():
    rf_mf = RandomForestRegressor(
        n_estimators=200, max_depth=None, max_features=mf,
        oob_score=True, random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_mf.fit(X_train, y_train)
    test_r = rmse(y_test, rf_mf.predict(X_test))
    oob_r  = rmse(y_train, rf_mf.oob_prediction_)
    mf_results[label] = {'test': test_r, 'oob': oob_r}
    print(f'{label:<35} Test RMSE = {test_r:.2f}$k  |  OOB RMSE = {oob_r:.2f}$k')

# Visualise
labels_mf = list(mf_results.keys())
test_vals  = [mf_results[l]['test'] for l in labels_mf]
oob_vals   = [mf_results[l]['oob']  for l in labels_mf]

x_pos = np.arange(len(labels_mf))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x_pos - width/2, test_vals, width, label='Test RMSE', color='#3498db', edgecolor='white')
bars2 = ax.bar(x_pos + width/2, oob_vals,  width, label='OOB RMSE',  color='#e67e22', edgecolor='white', alpha=0.85)

for bar, val in zip(bars1, test_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1, f'{val:.1f}', ha='center', fontsize=9)
for bar, val in zip(bars2, oob_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1, f'{val:.1f}', ha='center', fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels(labels_mf, rotation=10, ha='right', fontsize=9)
ax.set_ylabel('RMSE ($k)')
ax.set_title('max_features Effect on Random Forest Performance\n'
             'Fewer features per split = more decorrelated trees = lower variance',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print()
print('Note: Using ALL features (rightmost) = plain bagging. RMSE is higher.')
print('sqrt/log2 force more decorrelation → lower ensemble variance.')

In [ ]:
# ============================================================
# Cell 10: Tree Correlation Visualisation
#   sqrt features vs ALL features — show pairwise correlations
# ============================================================
N_TREES_VIZ = 20  # use first 20 trees for clear heatmap

# RF with sqrt features
rf_sqrt = RandomForestRegressor(
    n_estimators=N_TREES_VIZ, max_depth=None, max_features='sqrt',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_sqrt.fit(X_train, y_train)

# RF with all features (plain bagging)
rf_all = RandomForestRegressor(
    n_estimators=N_TREES_VIZ, max_depth=None, max_features=None,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_all.fit(X_train, y_train)

# Get per-tree test predictions
def get_per_tree_preds(rf_model, X):
    """Return predictions from each tree individually."""
    return np.array([tree.predict(X) for tree in rf_model.estimators_])

preds_sqrt = get_per_tree_preds(rf_sqrt, X_test)  # shape (20, n_test)
preds_all  = get_per_tree_preds(rf_all,  X_test)

corr_sqrt = np.corrcoef(preds_sqrt)
corr_all  = np.corrcoef(preds_all)

mean_corr_sqrt = (corr_sqrt.sum() - N_TREES_VIZ) / (N_TREES_VIZ * (N_TREES_VIZ - 1))
mean_corr_all  = (corr_all.sum()  - N_TREES_VIZ) / (N_TREES_VIZ * (N_TREES_VIZ - 1))

print(f'Mean pairwise tree correlation (sqrt features): {mean_corr_sqrt:.4f}')
print(f'Mean pairwise tree correlation (all features):  {mean_corr_all:.4f}')
print(f'Correlation reduction from feature subsampling: {100*(mean_corr_all - mean_corr_sqrt)/mean_corr_all:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Pairwise Tree Prediction Correlations\n'
             f'(First {N_TREES_VIZ} trees, tested on {len(y_test)} houses)',
             fontsize=13, fontweight='bold')

for ax, cm, title, mean_c in [
    (axes[0], corr_sqrt,
     f'Random Forest (max_features=sqrt)\nMean ρ = {mean_corr_sqrt:.3f}', mean_corr_sqrt),
    (axes[1], corr_all,
     f'Plain Bagging (max_features=all)\nMean ρ = {mean_corr_all:.3f}', mean_corr_all),
]:
    mask = np.eye(N_TREES_VIZ, dtype=bool)
    sns.heatmap(cm, ax=ax, vmin=0.5, vmax=1.0, cmap='YlOrRd',
                square=True, linewidths=0.3,
                cbar_kws={'shrink': 0.7, 'label': 'Pearson ρ'})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Tree index')
    ax.set_ylabel('Tree index')

plt.tight_layout()
plt.show()

print()
print('Random feature subsets (sqrt) produce noticeably lower inter-tree correlations.')
print('Lower ρ → lower ensemble variance floor → better generalisation.')

In [ ]:
# ============================================================
# Cell 11: Bootstrap Effect — Single Tree vs RF Variance
#   Run single tree 50 times (different seeds) → large spread
#   Run RF 50 times (different seeds) → small spread
# ============================================================
N_TRIALS = 50
single_tree_preds_trial = []
rf_preds_trial          = []

# Use ONE fixed test house for tracking
test_house_idx = 0

for seed in range(N_TRIALS):
    X_tr, X_te, y_tr, _ = train_test_split(X, y, test_size=0.2, random_state=seed)

    # Single tree
    t = DecisionTreeRegressor(max_depth=None, random_state=seed)
    t.fit(X_tr, y_tr)
    single_tree_preds_trial.append(t.predict(X_te[:1])[0])

    # Random forest
    rf_t = RandomForestRegressor(
        n_estimators=50, max_depth=None, max_features='sqrt',
        random_state=seed, n_jobs=-1
    )
    rf_t.fit(X_tr, y_tr)
    rf_preds_trial.append(rf_t.predict(X_te[:1])[0])

std_single = np.std(single_tree_preds_trial)
std_rf     = np.std(rf_preds_trial)

print('Prediction Variance Over 50 Different Random Train/Test Splits:')
print(f'  Single Tree std:     ${std_single:.1f}k')
print(f'  Random Forest std:   ${std_rf:.1f}k')
print(f'  Variance reduction:  {100*(1 - std_rf/std_single):.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Prediction Stability: Single Tree vs Random Forest (50 Trials)',
             fontsize=13, fontweight='bold')

for ax, preds, title, color in [
    (axes[0], single_tree_preds_trial, f'Single Tree\nStd = ${std_single:.1f}k', '#e74c3c'),
    (axes[1], rf_preds_trial,           f'Random Forest (50 trees)\nStd = ${std_rf:.1f}k', '#27ae60')
]:
    ax.hist(preds, bins=18, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(np.mean(preds), color='black', lw=2, linestyle='--',
               label=f'Mean = {np.mean(preds):.0f}k')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Predicted Price ($k)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('The Random Forest distribution is much tighter — your prediction is more reliable.')

In [ ]:
# ============================================================
# Cell 12: Feature Importance
#   Impurity-based (MDI) vs Permutation Importance
# ============================================================
rf_fi = RandomForestRegressor(
    n_estimators=200, max_depth=None, max_features='sqrt',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_fi.fit(X_train, y_train)

# MDI feature importance
mdi_importances = rf_fi.feature_importances_

# Permutation importance (on test set)
perm_result = permutation_importance(
    rf_fi, X_test, y_test,
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)
perm_importances = perm_result.importances_mean

# Sort by MDI
sort_idx = np.argsort(mdi_importances)[::-1]
feat_names_sorted = [FEATURES[i] for i in sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance: Impurity-based vs Permutation', fontsize=13, fontweight='bold')

colors_fi = sns.color_palette('viridis', len(FEATURES))

# MDI
axes[0].barh(feat_names_sorted, mdi_importances[sort_idx],
             color=[colors_fi[i] for i in range(len(FEATURES))], edgecolor='white')
axes[0].set_xlabel('Mean Decrease in Impurity (MDI)')
axes[0].set_title('Impurity-Based Importance\n(Mean Decrease Impurity)', fontsize=11)
axes[0].invert_yaxis()

# Permutation (sort by permutation importance)
perm_sort_idx = np.argsort(perm_importances)[::-1]
perm_names_sorted = [FEATURES[i] for i in perm_sort_idx]
xerr = perm_result.importances_std[perm_sort_idx]

axes[1].barh(perm_names_sorted, perm_importances[perm_sort_idx],
             xerr=xerr, color=[colors_fi[i] for i in range(len(FEATURES))],
             edgecolor='white', capsize=4)
axes[1].set_xlabel('Increase in RMSE when feature is shuffled')
axes[1].set_title('Permutation Importance\n(More reliable for correlated features)', fontsize=11)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('Feature ranking comparison:')
print(f'{"Feature":<16} {"MDI Rank":<12} {"Permutation Rank"}')
print('-' * 42)
mdi_ranks  = {FEATURES[i]: r+1 for r, i in enumerate(sort_idx)}
perm_ranks = {FEATURES[i]: r+1 for r, i in enumerate(perm_sort_idx)}
for f in FEATURES:
    print(f'{f:<16} {mdi_ranks[f]:<12} {perm_ranks[f]}')

In [ ]:
# ============================================================
# Cell 13: Correlation-Variance Trade-off Visualisation
#   Theoretical plot: ensemble variance floor as function of rho
#   with markers for "all features" and "sqrt features"
# ============================================================
B_large = 500  # large B so (1-rho)*sigma^2/B ≈ 0
rho_values = np.linspace(0, 1, 200)
sigma2 = 1.0

var_floor = rho_values * sigma2  # limiting variance as B → inf

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rho_values, var_floor, color='#2c3e50', linewidth=2.5,
        label='Variance floor = ρσ²  (as B→∞)')
ax.fill_between(rho_values, 0, var_floor, alpha=0.12, color='#2c3e50')

# Annotate with approximate rho values from our experiment
ax.axvline(mean_corr_sqrt, color='#27ae60', linestyle='--', linewidth=2,
           label=f'RF sqrt features: ρ ≈ {mean_corr_sqrt:.3f}')
ax.axvline(mean_corr_all, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Bagging all features: ρ ≈ {mean_corr_all:.3f}')

ax.scatter([mean_corr_sqrt], [mean_corr_sqrt * sigma2], s=120, color='#27ae60', zorder=5)
ax.scatter([mean_corr_all],  [mean_corr_all  * sigma2], s=120, color='#e74c3c', zorder=5)

ax.annotate(f'RF (sqrt)\nFloor = {mean_corr_sqrt:.3f}σ²',
            xy=(mean_corr_sqrt, mean_corr_sqrt),
            xytext=(mean_corr_sqrt - 0.18, mean_corr_sqrt + 0.08),
            fontsize=10, color='#27ae60',
            arrowprops=dict(arrowstyle='->', color='#27ae60'))
ax.annotate(f'Bagging (all)\nFloor = {mean_corr_all:.3f}σ²',
            xy=(mean_corr_all, mean_corr_all),
            xytext=(mean_corr_all + 0.03, mean_corr_all + 0.10),
            fontsize=10, color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c'))

ax.set_xlabel('Average Pairwise Tree Correlation (ρ)', fontsize=12)
ax.set_ylabel('Ensemble Variance Floor (as B→∞)', fontsize=12)
ax.set_title('Why Random Feature Subsets Help\n'
             'Lower ρ = lower variance floor = better ensemble', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print('Random feature subsets shift the operating point left along this curve.')
print(f'Variance floor reduction: {100*(mean_corr_all - mean_corr_sqrt)/mean_corr_all:.1f}% lower with sqrt features.')

In [ ]:
# ============================================================
# Cell 14: max_depth Effect — Bias-Variance Trade-off in RF
# ============================================================
depth_values = [1, 2, 3, 5, 8, 12, None]
depth_labels = [str(d) if d is not None else 'None\n(full)' for d in depth_values]

depth_train_rmses = []
depth_test_rmses  = []

for depth in depth_values:
    rf_d = RandomForestRegressor(
        n_estimators=100, max_depth=depth, max_features='sqrt',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_d.fit(X_train, y_train)
    depth_train_rmses.append(rmse(y_train, rf_d.predict(X_train)))
    depth_test_rmses.append(rmse(y_test, rf_d.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(depth_values))
ax.plot(x, depth_train_rmses, marker='o', color='#3498db', linewidth=2.2,
        markersize=7, label='Train RMSE')
ax.plot(x, depth_test_rmses, marker='s', color='#e74c3c', linewidth=2.2,
        markersize=7, label='Test RMSE')
ax.set_xticks(x)
ax.set_xticklabels(depth_labels)
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('RMSE ($k)', fontsize=12)
ax.set_title('max_depth Effect in Random Forest\n'
             'Very shallow trees = high bias; deep trees = can overfit', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Annotate gap
best_test_i = np.argmin(depth_test_rmses)
ax.axvline(best_test_i, color='#27ae60', linestyle=':', linewidth=2, alpha=0.7,
           label=f'Best test: depth={depth_labels[best_test_i]}')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Random forests are typically run with full-depth trees (max_depth=None).')
print('The ensemble averaging controls variance — individual tree depth controls bias.')
print('Tip: if RF overfits, try max_depth=10–20 before reducing n_estimators.')

In [ ]:
# ============================================================
# Cell 15: Final Comprehensive Performance Table
# ============================================================
rf_final = RandomForestRegressor(
    n_estimators=200, max_depth=None, max_features='sqrt',
    oob_score=True, random_state=RANDOM_STATE, n_jobs=-1
)
rf_final.fit(X_train, y_train)

final_results = {
    'Single Tree (full depth)': rmse_single,
    'Bagging (max_features=all, 200 trees)': rmse(y_test, bag.predict(X_test)),
    'RF Scratch (sqrt, 100 trees)': rmse_scratch,
    'sklearn RF (sqrt, 200 trees)': rmse(y_test, rf_final.predict(X_test)),
}

fig, ax = plt.subplots(figsize=(10, 4))
color_map = ['#e74c3c', '#e67e22', '#3498db', '#27ae60']
bars = ax.barh(list(final_results.keys()), list(final_results.values()),
               color=color_map, edgecolor='white', height=0.5)

for bar, val in zip(bars, final_results.values()):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}$k', va='center', fontsize=10)

ax.set_xlabel('Test RMSE ($k) — Lower is Better', fontsize=11)
ax.set_title('Final Performance Comparison — House Price Prediction', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('='*65)
print(f'{"Method":<45} {"RMSE":>8}   Improvement')
print('='*65)
baseline = rmse_single
for name, r in final_results.items():
    print(f'{name:<45} {r:>6.2f}$k   {100*(baseline-r)/baseline:+.1f}%')
print('='*65)
oob_rmse_final = rmse(y_train, rf_final.oob_prediction_)
print(f'\nOOB RMSE (free validation): {oob_rmse_final:.2f}$k')
print(f'Test RMSE (held-out):       {list(final_results.values())[-1]:.2f}$k')

## Key Takeaways

1. **Random Forest = Bagging + Random Feature Subsets.** The feature subsets are the crucial addition: they force trees to find different structures, reducing inter-tree correlation, which lowers the ensemble variance floor.

2. **The variance floor is $\rho \sigma^2$.** Bagging drives $\sigma^2$ down (averaging). Random feature subsets drive $\rho$ down. Both matter; the product determines how good your ensemble can be.

3. **More trees never hurts.** Unlike boosting, adding trees to a random forest never increases variance. The only cost is compute time. Keep adding trees until the OOB error plateaus.

4. **OOB error is free cross-validation.** Because each tree is validated on the ~37% of samples it never saw, you get a reliable estimate of generalisation error without a separate holdout set. Use it to monitor training and tune hyperparameters.

5. **max_features is the most important hyperparameter.** The default (`sqrt` for classification, `max_features/3` for regression) works well, but tuning it can improve performance. Lower values increase diversity (lower $\rho$) at the cost of weaker individual trees (higher $\sigma^2$).

6. **Feature importance scores are practically useful.** MDI importances are fast but can bias toward high-cardinality features. Permutation importances are more robust and should be preferred for reliable feature selection.

7. **Random forests require minimal tuning.** The three main hyperparameters: `n_estimators` (set high), `max_features` (tune around sqrt), `max_depth` (usually None). This is a major practical advantage over gradient boosting.

## Self-Check Questions

---

**Question 1:**
Adding more trees to a random forest always reduces or maintains variance — it never increases it. Boosting is different: more boosting rounds *can* increase variance. Why does this asymmetry exist?

<details>
<summary>Show Answer</summary>

**In a random forest, each new tree is independent.** Its predictions are averaged with all previous trees. The ensemble variance formula is:
$$\text{Var}_{\text{ensemble}} = \rho \sigma^2 + \frac{(1-\rho)\sigma^2}{B}$$
As $B$ increases, the second term $(1-\rho)\sigma^2/B$ decreases monotonically. The first term $\rho \sigma^2$ is unaffected by $B$. So ensemble variance can only stay flat or decrease — never increase.

**In boosting, each new tree is not averaged — it is added to the previous ensemble as a correction term.** With each additional round, the combined model becomes more complex (higher effective model complexity). At some point, the boosted ensemble has enough capacity to memorise the training noise. When it does, the residuals the next tree is trained on contain noise, and the tree learns that noise. The overall model starts to overfit: training error keeps falling but test error rises. This is the classic high-variance regime.

In practice, boosting is controlled with a learning rate (shrinkage) and early stopping — neither of which is needed for random forests.
</details>

---

**Question 2:**
OOB error uses the ~37% of samples not in each tree's bootstrap. Why is this a valid, approximately unbiased estimate of generalisation error?

<details>
<summary>Show Answer</summary>

**Because each OOB prediction is made by a model that was never trained on that observation.**

For training point $x_i$, the OOB prediction is the average prediction from all trees that did not include $x_i$ in their bootstrap sample (roughly B/e ≈ 0.368B trees). These trees have never seen $y_i$ during training. Their predictions on $x_i$ are therefore equivalent to predictions from a model trained on a dataset that excludes $x_i$ — exactly the leave-one-out cross-validation condition.

The OOB estimate is slightly pessimistic (each OOB tree was trained on ~63% of the data, not 100%), but this pessimism is small and consistent. Studies have shown OOB error closely tracks 5-fold and 10-fold CV error. The key property that makes it valid: **the test observation was never used to fit the model making that prediction** — the fundamental requirement for an unbiased generalisation estimate.
</details>

---

**Question 3:**
Random forests use `max_features = sqrt(p)` by default. If you increase `max_features` to `p` (all features, plain bagging), how does tree correlation change? How does ensemble variance change? Is the final model better or worse?

<details>
<summary>Show Answer</summary>

**Tree correlation increases; ensemble variance increases; the model is typically worse.**

With all features available at every split, every tree will tend to split on the same most-predictive features (e.g., `sqft` in our dataset). Despite different bootstrap samples, the decision boundaries are similar. Pairwise tree correlation $\rho$ rises.

From the formula:
$$\text{Var}_{\text{ensemble}} = \rho \sigma^2 + \frac{(1-\rho)\sigma^2}{B}$$

- $\rho$ increases → first term rises
- $\sigma^2$ decreases slightly (each individual tree is stronger with access to all features)
- Net effect: the increase in $\rho$ typically outweighs the decrease in $\sigma^2$, so ensemble variance increases

The limiting variance floor is $\rho\sigma^2$, which is strictly higher under all-feature splits for problems where one feature dominates. In our code experiment, plain bagging (max_features=all) consistently produced higher test RMSE than random feature subsets. The `sqrt(p)` default is specifically chosen to balance the $\rho$ vs $\sigma^2$ trade-off for typical real-world datasets.
</details>

In [ ]:
# ============================================================
# Cell 16: Random Forest Hyperparameter Summary
# ============================================================
hparam_data = {
    'Hyperparameter': [
        'n_estimators', 'max_features', 'max_depth',
        'min_samples_split', 'min_samples_leaf', 'bootstrap', 'oob_score'
    ],
    'Default': [
        '100', 'sqrt(p)', 'None (full)',
        '2', '1', 'True', 'False'
    ],
    'Effect (increase ↑)': [
        'Lower variance (diminishing returns)', 'Higher variance per tree, lower ρ → lower ensemble variance',
        'Lower bias, higher variance per tree',
        'Higher bias (simpler trees)', 'Higher bias (simpler trees)',
        'Diversifies trees (critical)', 'Enables free validation error'
    ],
    'Tune when': [
        'Always (set high)', 'Trees too similar or too weak',
        'RF still overfits', 'Very large dataset (speed)',
        'Noisy data / imbalanced leaves', 'Always True', 'Always True'
    ]
}

df_hp = pd.DataFrame(hparam_data)
print('=== Random Forest Hyperparameter Guide ===')
print(df_hp.to_string(index=False))
df_hp

In [ ]:
# ============================================================
# Cell 17: Final Summary Print
# ============================================================
print('=' * 65)
print('NOTEBOOK 6 COMPLETE: Random Forests')
print('=' * 65)
print()
print('Concepts covered:')
print('  1.  Dataset: 1000 samples, 8 features, house price prediction')
print('  2.  RandomForestScratch: bootstrap + random feature subsets')
print('  3.  Comparison: Single Tree → Bagging → RF Scratch → sklearn RF')
print('  4.  n_estimators curve (1–300): stabilisation demonstration')
print('  5.  Out-of-Bag error: free validation, tracks CV closely')
print('  6.  OOB tracking over n_estimators: stopping criterion')
print('  7.  max_features grid: sqrt / log2 / 0.3 / 0.5 / all')
print('  8.  Tree correlation heatmap: sqrt vs all features')
print('  9.  Variance reduction: 50-trial comparison tree vs RF')
print('  10. Feature importance: MDI vs Permutation')
print('  11. Variance floor visualisation: ρ·σ² annotated with data')
print('  12. max_depth grid: bias-variance trade-off')
print('  13. Hyperparameter guide table')
print()
print('Next: Notebook 7 — Gradient Boosting')
print('  Boosting attacks bias; XGBoost/LightGBM push this to the limit')